# Oxford-IIIT Pet Dataset Exploration

This notebook is designed for Kaggle and covers the required dataset exploration:
- locate and load the Oxford-IIIT Pet images and segmentation trimaps;
- convert boundary/unclassified trimap pixels to foreground;
- inspect classes, class counts, species, image sizes, mask values, and foreground area;
- plot class-distribution bar charts; and
- display nine random images in a 3x3 grid with mask overlays.

Attach the Oxford-IIIT Pet dataset in Kaggle using **Add Input**. The code searches `/kaggle/input` automatically. If it cannot find the data, it downloads the dataset with `torchvision` into `/kaggle/working/pet_data` (Internet must be enabled). You can also set `DATASET_ROOT` manually.

In [ ]:
# Uncomment this only if Kaggle does not already provide these packages.
# %pip install pillow matplotlib pandas seaborn

from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

# Leave as None for automatic Kaggle discovery. Examples:
# DATASET_ROOT = Path('/kaggle/input/oxford-iiit-pet')
# DATASET_ROOT = Path('/kaggle/input/oxford-iiit-pet-dataset')
DATASET_ROOT = None

INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

def locate_dataset():
    roots = [Path(DATASET_ROOT)] if DATASET_ROOT else [p for p in INPUT_ROOT.rglob('*') if p.is_dir()]
    roots = [INPUT_ROOT] + roots if INPUT_ROOT not in roots else roots
    for root in roots:
        image_dirs = [root / 'images', root / 'images' / 'images']
        mask_dirs = [root / 'annotations' / 'trimaps', root / 'trimaps']
        for image_dir in image_dirs:
            for mask_dir in mask_dirs:
                if image_dir.is_dir() and mask_dir.is_dir():
                    return root, image_dir, mask_dir
    return None, None, None

DATASET_ROOT, IMAGE_DIR, MASK_DIR = locate_dataset()
if IMAGE_DIR is None:
    print('Dataset not found in /kaggle/input; downloading with torchvision (Internet required)...')
    from torchvision.datasets import OxfordIIITPet
    DOWNLOAD_ROOT = Path('/kaggle/working/pet_data') if Path('/kaggle/working').exists() else Path('./pet_data')
    DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    OxfordIIITPet(root=str(DOWNLOAD_ROOT), download=True)
    downloaded_base = DOWNLOAD_ROOT / 'oxford-iiit-pet'
    DATASET_ROOT = downloaded_base
    IMAGE_DIR = downloaded_base / 'images'
    MASK_DIR = downloaded_base / 'annotations' / 'trimaps'
    if not IMAGE_DIR.is_dir() or not MASK_DIR.is_dir():
        raise FileNotFoundError(f'Download completed, but expected folders were not found under {downloaded_base}')
print('Dataset root:', DATASET_ROOT.resolve())
print('Image directory:', IMAGE_DIR.resolve())
print('Mask directory:', MASK_DIR.resolve())

## Build the Image/Mask Index

Oxford-IIIT Pet image names have the form `Breed_name_001.jpg`; the numeric suffix identifies the individual image and the preceding text identifies the breed. Raw trimaps normally contain values 1 (foreground), 2 (background), and 3 (boundary).

In [ ]:
image_files = sorted(p for p in IMAGE_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS)
mask_files = {p.stem: p for p in MASK_DIR.rglob('*.png')}

def breed_from_stem(stem):
    return re.sub(r'_\d+$', '', stem).replace('_', ' ').title()

rows = []
for image_path in image_files:
    mask_path = mask_files.get(image_path.stem)
    if mask_path is not None:
        rows.append({'image_path': image_path, 'mask_path': mask_path,
                     'breed': breed_from_stem(image_path.stem)})

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('No image/mask pairs were found. Check DATASET_ROOT and folder names.')

# The 12 known cat breeds are used to derive the species column.
CAT_BREEDS = {'Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair',
              'Egyptian Mau', 'Maine Coon', 'Persian', 'Ragdoll', 'Russian Blue',
              'Siamese', 'Sphynx'}
df['species'] = np.where(df['breed'].isin(CAT_BREEDS), 'Cat', 'Dog')
df['split'] = 'all'

# Use official split files when supplied by the downloaded dataset.
for split_name, split_file in [('trainval', 'trainval.txt'), ('test', 'test.txt')]:
    candidates = list(DATASET_ROOT.rglob(split_file))
    if candidates:
        names = set()
        for line in candidates[0].read_text().splitlines():
            if line.strip() and not line.startswith('#'):
                names.add(line.split()[0])
        df.loc[df['image_path'].map(lambda p: p.stem).isin(names), 'split'] = split_name

print(f'Paired samples: {len(df):,}')
print(f'Breeds/classes: {df.breed.nunique()}')
display(df.head())

In [ ]:
# Basic dataset summary and class/species balance.
summary = pd.DataFrame({
    'value': [len(df), df['breed'].nunique(), df['species'].nunique(),
              df['image_path'].map(lambda p: p.suffix.lower()).nunique()],
}, index=['paired samples', 'breeds/classes', 'species', 'image extensions'])
display(summary)
print('Samples per split:')
display(df['split'].value_counts().rename_axis('split').to_frame('samples'))
print('Samples per species:')
display(df['species'].value_counts().rename_axis('species').to_frame('samples'))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
breed_counts = df['breed'].value_counts().sort_values()
breed_counts.plot(kind='barh', ax=axes[0], color='#4c78a8')
axes[0].set_title('Images per breed/class')
axes[0].set_xlabel('Number of paired images')
df['species'].value_counts().plot(kind='bar', ax=axes[1], color=['#f58518', '#54a24b'])
axes[1].set_title('Images per species')
axes[1].set_ylabel('Number of paired images')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect image dimensions and raw trimap values.
dimension_rows, mask_value_counts, foreground_ratios = [], [], []
for row in df.itertuples(index=False):
    with Image.open(row.image_path) as im:
        width, height = im.size
    mask = np.array(Image.open(row.mask_path))
    dimension_rows.append((width, height))
    values, counts = np.unique(mask, return_counts=True)
    mask_value_counts.extend(zip(values.tolist(), counts.tolist()))
    foreground_ratios.append(np.isin(mask, [1, 3]).mean())
df[['width', 'height']] = pd.DataFrame(dimension_rows, index=df.index)
df['foreground_ratio'] = foreground_ratios
print('Image dimension distribution:')
display(df.groupby(['width', 'height']).size().sort_values(ascending=False).head(15).rename('images').to_frame())

raw_mask_counts = pd.DataFrame(mask_value_counts, columns=['raw_value', 'pixels']).groupby('raw_value')['pixels'].sum()
raw_mask_counts.index = raw_mask_counts.index.map({1: 'foreground (1)', 2: 'background (2)', 3: 'boundary (3)'}.get)
print('Raw trimap pixel values across all masks:')
display(raw_mask_counts.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['foreground_ratio'], bins=20, kde=True, ax=axes[0], color='#e45756')
axes[0].set_title('Foreground ratio after {1, 3} -> foreground')
axes[0].set_xlabel('Foreground pixels / total pixels')
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=SEED), x='width', y='height', hue='species', alpha=0.55, ax=axes[1])
axes[1].set_title('Image dimensions')
plt.tight_layout()
plt.show()

## Mask Conversion

For binary pet segmentation, background remains 0 and both raw foreground (1) and boundary/unclassified (3) become 1. This is the conversion requested for the masks.

In [ ]:
def load_binary_pet_mask(mask_path):
    raw_mask = np.array(Image.open(mask_path))
    # Raw trimap: 1=pet, 2=background, 3=boundary/unclassified.
    return np.isin(raw_mask, [1, 3]).astype(np.uint8)

example_mask = load_binary_pet_mask(df.iloc[0]['mask_path'])
print('Converted mask values:', np.unique(example_mask), '(0=background, 1=pet)')
print('Converted foreground ratio:', round(example_mask.mean(), 4))

## Required 3x3 Random Visualization

Green pixels show the converted pet foreground mask over each original image. A fixed seed makes the nine examples reproducible; change `SEED` to see another sample.

In [ ]:
sample = df.sample(n=min(9, len(df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, row in zip(axes.flat, sample.itertuples(index=False)):
    image = np.array(Image.open(row.image_path).convert('RGB'))
    mask = load_binary_pet_mask(row.mask_path)
    if mask.shape != image.shape[:2]:
        mask = np.array(Image.fromarray(mask).resize((image.shape[1], image.shape[0]), Image.Resampling.NEAREST))
    overlay = image.copy()
    overlay[mask == 1] = (0.55 * overlay[mask == 1] + 0.45 * np.array([40, 210, 80])).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f'{row.breed} ({row.species})', fontsize=10)
    ax.axis('off')
for ax in axes.flat[len(sample):]:
    ax.axis('off')
plt.suptitle('Nine random Oxford-IIIT Pet images with binary mask overlays', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: compare raw trimaps with converted binary masks.
mask_sample = df.sample(n=min(4, len(df)), random_state=SEED + 1).reset_index(drop=True)
fig, axes = plt.subplots(len(mask_sample), 3, figsize=(12, 4 * len(mask_sample)))
axes = np.atleast_2d(axes)
for i, row in mask_sample.iterrows():
    image = Image.open(row.image_path).convert('RGB')
    raw = np.array(Image.open(row.mask_path))
    binary = load_binary_pet_mask(row.mask_path)
    axes[i, 0].imshow(image); axes[i, 0].set_title(row.breed + ' image')
    axes[i, 1].imshow(raw, cmap='viridis', vmin=1, vmax=3); axes[i, 1].set_title('Raw trimap')
    axes[i, 2].imshow(binary, cmap='gray', vmin=0, vmax=1); axes[i, 2].set_title('Binary mask')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.show()